1. feladat: Importáljuk be az adatelemzéshez szükséges könyvtárakat!

In [ ]:
# 1. feladat: Importalas
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

2. feladat: Olvassuk be a mellékelt spotify_yt.csv fájlt és tároljuk el egy DataFrame-ben a fájl tartalmát!

In [ ]:
df = pd.read_csv('spotify_yt.csv')
df.head()

3. feladat: Írassuk ki az adatkeretre vonatkozó statisztikákat és információkat!

In [ ]:
print('Shape:', df.shape)
print('\nInfo:')
df.info()
print('\nLeíró statisztika:')
df.describe()

4. feladat: Számoljuk meg, hogy oszloponként mennyi null érték fordul elő!

In [ ]:
print(df.isnull().sum())

5. Töltsük fel a hiányzó adatokat Loudness, Acousticness, Instrumentalness, Energy, Danceability oszlopokban!
Feltöltéskor vegyük figyelembe, hogy az előadóhoz tartozó számok milyen átlagot alkotnak az adott oszlopokban, és azzal töltsük fel a hiányzó értékeket!
Ellenőrizzük az előző feladat műveletével, hogy sikerült-e a feltöltés!

In [ ]:
columns_to_fill = ['Loudness', 'Acousticness', 'Instrumentalness', 'Energy', 'Danceability']
for col in columns_to_fill:
    artist_means = df.groupby('Artist')[col].transform('mean')
    df[col] = df[col].fillna(artist_means)
print(df[columns_to_fill].isnull().sum())

6. feladat: Írassuk ki azokat a sorokat, ahol a Speechiness, Liveness, Valence, Tempo, Key 
vagy Duration_ms oszlopok értékeiben null érték van! 
Kiíratás után dobjuk el ezeket a sorokat helyben, majd ellenőrizzük, hogy sikerült
e a null értékek eltávolítása az adott oszlopokból.

In [ ]:
columns_to_check = ['Speechiness', 'Liveness', 'Valence', 'Tempo', 'Key', 'Duration_ms']
null_rows = df[df[columns_to_check].isnull().any(axis=1)]
print('Null érték tartalmazó sorok száma:', len(null_rows))
print(null_rows)

df.dropna(subset=columns_to_check, inplace=True)

print('\nEllenőrzés a null értékek eltávolítása után:')
print(df[columns_to_check].isnull().sum())

7. feladat: Írassuk ki a DataFrame oszlopainak az adattípusait! 
Módosítsuk az adatszerkezetet úgy, hogy a Duration_ms, Key, Views, Likes, 
Comments, Stream oszlopok adattípusát int-re konvertáljuk, a Licensed, 
official_video oszlopok értékét boolean típusra konvertáljuk! 
Ha az érték null érték lenne, adjunk meg -1-et az inteknél, illetve False-t a 
boolean adatoknál!

In [ ]:
print('\nAdattípusok a konverzió előtt:')
print(df.dtypes)
int_columns = ['Duration_ms', 'Key', 'Views', 'Likes', 'Comments', 'Stream']
bool_columns = ['Licensed', 'official_video']

for col in int_columns:
    df[col] = df[col].fillna(-1).astype(int)

for col in bool_columns:
    df[col] = df[col].fillna(False).astype(bool)
    
print('\nAdattípusok a konverzió után:')
print(df.dtypes)

8.feladat: Dobjuk el azokat a sorokat, ahol a Stream oszlopban -1 található! Dobjuk el az Url_spotify, Uri oszlopokat is!

In [ ]:
df = df[df['Stream'] != -1]
df.drop(columns=['Url_spotify', 'Uri'], inplace=True)
print('Oszlopok eltávolítások után:', df.columns.tolist())

9. feladat: Mentsük el a tisztított adatokat egy spotify_yt_cleaned.csv nevű fájlba!

In [ ]:
df.to_csv('spotify_yt_cleaned.csv', index=False)

10. feladat: Számoljuk meg, hogy hány zeneszámhoz volt található Youtube videó!

In [ ]:
youtube_count = df['Url_youtube'].notnull().sum()
print('Zenék száma YouTube URL-lel:', youtube_count)

11. feladat: Keressük meg melyik az a 10 előadó, akinek a zenéjére lehetne a leginkább 
táncolni!  
A rangsor állításakor azt vegyük figyelembe, hogy átlagosan milyen Danceability 
értékei vannak az előadó számainak.

In [ ]:
top10_dance_artists = df.groupby('Artist')['Danceability'].mean().sort_values(ascending=False).head(10)
print('Top 10 Danceability előadó:')
print(top10_dance_artists)

12. feladat: Határozzuk meg és írassuk ki a 10 legtöbbet streamelt előadót a 10 legtöbbet 
nézett előadóval együtt! 
Ábrázoljuk egyetlen diagrammon, két tengellyel a 10 legtöbbet streamelt előadót 
és hasonlítsuk össze oszlopdiagrammokkal, hogy mennyi Youtube megtekintésük 
van a Streamek száma mellett!  
Figyeljünk a helyes megjelenítésre! Tegyük részben átlátszóvá az előbb álló 
oszlopot! 
Jelenítsünk meg jelmagyarázatot legalább az egyik oszlopra!

In [ ]:
top_streamed = df.groupby('Artist')['Stream'].sum().sort_values(ascending=False).head(10)
top_viewed = df.groupby('Artist')['Views'].sum().sort_values(ascending=False).head(10)
views = top_viewed.reindex(top_streamed.index).fillna(0)
print('Top 10 streamed artists:')
print(top_streamed)
print('\nTop 10 viewed artists:')
print(top_viewed)

fig, ax1 = plt.subplots(figsize=(12, 6))

width = 0.4
x = np.arange(len(top_streamed.index))

ax1.bar(x - width/2, top_streamed / 1e10, width, color='blue', label='Streams')
ax1.set_ylabel('Streams (e10)')
ax1.set_xticks(x)
ax1.set_xticklabels(top_streamed.index, rotation=45)

ax2 = ax1.twinx()
ax2.bar(x + width/2, views / 1e10, width, color='red', label='Views')
ax2.set_ylabel('Views (e10)')

fig.legend(loc='upper right')
plt.title('Top 10 Streamed Artists and Their YouTube Views')
plt.tight_layout()
plt.show()

13. feladat: Ábrázoljuk korrelációs mátrixxal, hogy milyen zenei tulajdonságok járultak hozzá 
a 10 legnépszerűbb zeneszámhoz (streamelések alapján!) 
Figyeljünk oda, hogy a számokat tartalmazó oszlopokkal dolgozzunk csak! 
Ügyeljünk a jól látható megjelenítésre!

In [ ]:
top_songs = df.sort_values('Stream', ascending=False).drop_duplicates(subset=['Track']).head(10)
numeric_cols = top_songs.select_dtypes(include=[np.number]).columns
corr_matrix = top_songs[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix for Top 10 Streamed Songs')
plt.show()

14. feladat: Döntsük el, hogy kinek van a legtöbb streamelése Spotify-on és megtekintése 
Youtube-on J. Cole, Kendrick Lamar és Drake közül a jelenlegi adathalmazunk 
alapján! 
A szűréskor figyeljünk arra, hogy az említett előadók zeneszámok nevében is 
feltűnhetnek és hogy ne legyenek duplikátumok az adataink között (tekinthetjük 
úgy, hogy a zeneszám neve egyedi). 
Ábrázoljuk az eredményeket egymás mellett két kördiagrammon, ahol 
megjelenítjük a neveket a körcikkek mellett, egyedi színeket adunk a 
körcikkeknek, majd nevet adunk a kördiagrammoknak! 
Állítsunk be tetszőleges hátteret az egész diagrammnak!

In [ ]:
artists = ['J. Cole', 'Kendrick Lamar', 'Drake']
filtered_df = df[df['Artist'].isin(artists)].drop_duplicates(subset='Track')
streams = filtered_df.groupby('Artist')['Stream'].sum()
views = filtered_df.groupby('Artist')['Views'].sum()

fig, axs = plt.subplots(1, 2, figsize=(12, 6))
fig.patch.set_facecolor('white')

streams.plot(kind='pie', ax=axs[0], autopct='%1.1f%%', labels=streams.index, colors=['red', 'lightgreen', 'lightblue'])
axs[0].set_title('Spotify Streams')

views.plot(kind='pie', ax=axs[1], autopct='%1.1f%%', labels=views.index, colors=['red', 'lightgreen', 'lightblue'])
axs[1].set_title('YouTube Views')

plt.show()